### setting up the llm model, setting up the embedding system

In [12]:
import os # environment setup
from langchain_groq import ChatGroq # llm model to be used
from dotenv import load_dotenv # To establish api key connection to the groq llm

from langchain_huggingface import HuggingFaceEmbeddings # for embedding chunks
from langchain_chroma import Chroma

load_dotenv()

# setup the llm
llm = ChatGroq(model= "llama-3.3-70b-versatile",
               temperature=0,
               api_key= os.getenv("GROQ_API_KEY"), # Optional if the env var is named exactly GROQ_API_KEY
                max_tokens=1024
               ) 

embeddings =  HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
CHUNK_SIZE = 350
CHUNK_OVERLAP = 50
COLLECTION_NAME = "legal-rag"
PERSIST_DIRECTORY = "./chroma_db"
DATA_DIRECTORY = r"C:\Users\user\Desktop\RAG Projects\Legal RAG 1\data\lawsuit.txt" 

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


### Document loader and processer

#### Loading

In [2]:
from typing import List, Dict
from pathlib import Path # for loading path
from langchain_community.document_loaders import PyMuPDFLoader, TextLoader, UnstructuredExcelLoader # for loading different types of documents
from langchain_text_splitters import RecursiveCharacterTextSplitter # splitting the text 

def load_documents(directory: str) -> List:
    documents = []
    dir_path = Path(directory)

    
    if not dir_path.exists():
        raise FileNotFoundError(f'the file {directory} does not exist')
    
    # NEW LOGIC: Create a list of files to process
    if dir_path.is_file():
        files_to_process = [dir_path] # Just this one file
    else:
        files_to_process = dir_path.glob('**/*') # All files in the folder

    for file_path in files_to_process:
        if not file_path.is_file():
            continue

        try:
            if file_path.suffix.lower() == '.pdf':
                loader = PyMuPDFLoader(str(file_path))
                documents.extend(loader.load())
            elif file_path.suffix.lower() == '.txt':
                loader = TextLoader(str(file_path))
                documents.extend(loader.load())
            elif file_path.suffix.lower() in ('.xlsx', '.xls'):
                loader = UnstructuredExcelLoader(str(file_path))
                documents.extend(loader.load())
            elif file_path.suffix.lower() in ('.doc', '.docx'):
                # Note: Docx usually uses UnstructuredWordDocumentLoader or Docx2txtLoader
                from langchain_community.document_loaders import Docx2txtLoader
                loader = Docx2txtLoader(str(file_path))
                documents.extend(loader.load())
            else:
                print(f"Skipping unsupported file: {file_path.name}")

        except Exception as e:
            # Print the error so you know which file failed, but keep going
            print(f"Error loading {file_path.name}: {str(e)}")

    return documents

#### Processing: splitting the documents 

In [3]:
def split_docs_into_chunks(documents: List, chunk_size = CHUNK_SIZE, chunk_overlap = CHUNK_OVERLAP) -> List:

    if not documents:
        print(f"No document provided to split")
    
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size = chunk_size,
        chunk_overlap = chunk_overlap,
        length_function = len,
        add_start_index = True,
    )

    processed_docs = text_splitter.split_documents(documents)
    return processed_docs

In [8]:
# testing

# 1. Define your path (create a folder named 'data' and put a PDF there)


# 2. Run the pipeline
raw_docs = load_documents(DATA_DIRECTORY)

# 3. Process into chunks using your config constants
chunks = split_docs_into_chunks(
    documents=raw_docs, 
    chunk_size=CHUNK_SIZE, 
    chunk_overlap=CHUNK_OVERLAP
)

# 4. Peek at a chunk
if chunks:
    print(f"\nExample Chunk Content:\n{chunks[0].page_content[:200]}...")
    print(f"\nMetadata: {chunks[0].metadata}")


Example Chunk Content:
Case Title: Smith v. Johnson

Case Number: 2024-CV-017

Court: United States District Court, Northern District of California

Date: March 10, 2024

Plaintiff: John Smith
Defendant: Michael Johnson...

Metadata: {'source': 'C:\\Users\\user\\Desktop\\RAG Projects\\Legal RAG 1\\data\\lawsuit.txt', 'start_index': 0}


### Vector Database

In [10]:
import os
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings

# initiate embeddings
# it automatically detects if you have gpu(cuda) or cpu
def get_embeddings_model():
    return HuggingFaceEmbeddings(
        model_name = "sentence-transformers/all-MiniLM-L6-v2"
    )

# creating vector store function
def create_vector_store(chunks, collection_name, persist_directory=PERSIST_DIRECTORY):

    """
    Takes text chunks and saves them into a local ChromaDB folder.
    """
    if not chunks:
        print("No chunks provided to store.")
        return None
    
    embeddings = get_embeddings_model()
    
    # This single line handles embedding and saving to disk
    vectorstore = Chroma.from_documents(
        documents=chunks,
        embedding=embeddings,
        collection_name=collection_name,
        persist_directory=persist_directory
    )
    
    print(f"Success: Documents stored at '{persist_directory}'")
    return vectorstore


# loading function 
def load_vector_store(collection_name, persist_directory= PERSIST_DIRECTORY):
    """
    Connects to an existing ChromaDB folder on your disk.
    """
    embeddings = get_embeddings_model()
    
    vectorstore = Chroma(
        collection_name=collection_name,
        embedding_function=embeddings,
        persist_directory=persist_directory
    )
    
    print(f"Loaded existing index from '{persist_directory}'")
    return vectorstore


### Document Ingestion Pipeline

Its job is to take raw files from your computer, break them into smaller pieces (chunks), convert those pieces into numbers (embeddings), and save them into a vector database so your chatbot can "read" them later.

In [11]:
import gc
import numpy as np
from typing import Optional
from langchain_core.documents import Document
import torch
# all other libraries and dependencies have been imported earlier

def run_ingestion_pipeline(data_path, collection_name=COLLECTION_NAME, persist_dir=PERSIST_DIRECTORY, clear_existing=False):
    """
    Orchestrates the full flow: Load -> Split -> Store.
    """
    print(f"Starting Ingestion Pipeline for: {data_path}")

    # 1. Load Documents
    # Your load_documents returns a tuple (documents, ) based on your snippet
    raw_docs_result = load_documents(data_path)
    
    # Handling the tuple return from your specific function definition
    if isinstance(raw_docs_result, tuple):
        raw_docs = raw_docs_result[0]
    else:
        raw_docs = raw_docs_result

    if not raw_docs:
        print("No documents found. Pipeline aborted.")
        return None

    # 2. Split into Chunks
    # Using your split_docs_into_chunks function
    print(f"Splitting {len(raw_docs)} documents into chunks...")
    chunks_result = split_docs_into_chunks(
        documents=raw_docs, 
        chunk_size=CHUNK_SIZE, 
        chunk_overlap=CHUNK_OVERLAP
    )
    
    # Handling the tuple return from your specific function definition
    chunks = chunks_result[0] if isinstance(chunks_result, tuple) else chunks_result

    # 3. Create Vector Store (ChromaDB)
    print(f"Creating embeddings and saving to ChromaDB (this may take a moment)...")
    
    # If you want to overwrite the DB, you'd manually delete the folder, 
    # but create_vector_store will append by default.
    vectorstore = create_vector_store(
        chunks=chunks,
        collection_name=collection_name,
        persist_directory=persist_dir
    )

    # 4. Memory Cleanup
    # Important for Notebooks to prevent kernel crashes/RAM spikes
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    print("\n✅ Ingestion Pipeline Complete!")
    return vectorstore

# --- RUNNING THE PIPELINE ---
# You can point this to a folder or a single file
TARGET_PATH = r"C:\Users\user\Desktop\RAG Projects\Legal RAG 1\data" 

# Execute
vectorstore = run_ingestion_pipeline(TARGET_PATH)

Starting Ingestion Pipeline for: C:\Users\user\Desktop\RAG Projects\Legal RAG 1\data
Error loading memo_2_test.docx: No module named 'docx2txt'
Splitting 1 documents into chunks...
Creating embeddings and saving to ChromaDB (this may take a moment)...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Success: Documents stored at './chroma_db'

✅ Ingestion Pipeline Complete!


In [18]:
!pip install langchain-classic

### The Legal RAG chatbot

In [24]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

# 1. SETUP GLOBAL MEMORY
store = {}

def get_session_history(session_id: str):
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    return store[session_id]

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

def get_legal_chat_chain(vectorstore):
    global llm 
    
    system_prompt = (
        "You are a precise Legal Assistant. Answer based ONLY on the context.\n\n"
        "Context:\n{context}"
    )
    
    prompt = ChatPromptTemplate.from_messages([
        ("system", system_prompt),
        MessagesPlaceholder(variable_name="chat_history"),
        ("human", "{input}"),
    ])

    retriever = vectorstore.as_retriever(search_kwargs={"k": 4})
    
    # FIX: Use RunnablePassthrough to assign variables properly
    rag_chain = (
        RunnablePassthrough.assign(
            context=lambda x: format_docs(retriever.invoke(x["input"]))
        )
        | prompt
        | llm
        | StrOutputParser()
    )

    return RunnableWithMessageHistory(
        rag_chain,
        get_session_history,
        input_messages_key="input",
        history_messages_key="chat_history",
    )

def ask_legal_assistant(query, vectorstore):
    if not query.strip(): return "Ask a question."
    
    chain = get_legal_chat_chain(vectorstore)
    config = {"configurable": {"session_id": "notebook_session"}}
    
    try:
        # In LCEL, we get the string directly from the StrOutputParser
        answer = chain.invoke({"input": query}, config=config)
        return answer
    except Exception as e:
        return f"⚠️ Error: {str(e)}"

def reset_chat():
    store.clear()
    print("✨ Chat history cleared.")

In [25]:
# Assuming your vectorstore (e.g., ChromaDB) is already initialized as 'vectorstore'
query = "Who is the plaintiff in the case Smith v. Johnson?"
response = ask_legal_assistant(query, vectorstore)
print(response)

The plaintiff in the case Smith v. Johnson is John Smith.
